# C_2: Transformer Embeddings

Fitur: FinBERT / IndoBERT / LLM (tugas 1a-ii). Data dan pooling sama dengan C_1 supaya bisa dibandingkan langsung.

In [16]:
from c_common import *

D, text = load_all()
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 9)


In [17]:
#we get finBERT ya
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert").eval()

def embed(texts, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], padding=True, truncation=True,
                  max_length=512, return_tensors="pt")
        with torch.no_grad():
            h = model(**enc).last_hidden_state[:, 0]   # [CLS] vector, 768 dims
        out.append(h.numpy())
    return np.vstack(out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 256.71it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
texts = ["The central bank raised interest rates, strengthening the dollar.",
         "Oil prices surged after the escalation in the Middle East."]
enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
with torch.no_grad():
    out = model(**enc)

emb = out.last_hidden_state[:, 0]      
print(emb.shape)                       
print(torch.isnan(emb).any())  

torch.Size([2, 768])
tensor(False)


In [19]:
import torch.nn.functional as F

s = ["The dollar strengthened against the rupiah.",
     "The USD rose versus the Indonesian rupiah.",
     "The recipe needs two cups of flour."]
enc = tok(s, padding=True, return_tensors="pt")
with torch.no_grad():
    e = model(**enc).last_hidden_state[:, 0]
print(F.cosine_similarity(e[0:1], e[1:2]))   
print(F.cosine_similarity(e[0:1], e[2:3])) 

tensor([0.6919])
tensor([-0.1528])


## Uji 2 hari nyata dan perbandingan pooling

Sebelum meng-embed semua hari, cek dulu pada 2 hari pertama: bentuk keluaran, ada tidaknya NaN, dan perbedaan pooling `[CLS]` vs rata-rata token.

In [20]:
# Uji pada 2 hari nyata: bentuk, NaN, dan perbandingan pooling CLS vs mean.
# Nama kolom teks tergantung corpus di load_all(): "filtered" -> judul_gabungan, "id" -> judul_gabungan_en.
TITLE = "judul_gabungan" if "judul_gabungan" in text.columns else "judul_gabungan_en"
LEAD = "lead_paragraph_gabungan" if "lead_paragraph_gabungan" in text.columns else "lead_paragraph_gabungan_en"

t2 = text.head(2)
t2_txt = (t2[TITLE].fillna("") + ". " + t2[LEAD].fillna("")).tolist()
print("kolom teks:", TITLE, "+", LEAD)
print("tanggal:", [d.date() for d in t2.index])

enc = tok(t2_txt, padding=True, truncation=True, max_length=512, return_tensors="pt")
print("panjang token:", [len(tok(x)["input_ids"]) for x in t2_txt], "(maks 512, lebih dari itu dipotong)")
with torch.no_grad():
    h = model(**enc).last_hidden_state
m = enc["attention_mask"].unsqueeze(-1)
cls_v = h[:, 0]
mean_v = (h * m).sum(1) / m.sum(1)          # rata-rata token, padding diabaikan

print("bentuk CLS:", tuple(cls_v.shape), "| mean:", tuple(mean_v.shape))
print("ada NaN:", bool(torch.isnan(cls_v).any() or torch.isnan(mean_v).any()))
print("cos(hari1, hari2)  CLS:", round(F.cosine_similarity(cls_v[0:1], cls_v[1:2]).item(), 3),
      "| mean:", round(F.cosine_similarity(mean_v[0:1], mean_v[1:2]).item(), 3))

kolom teks: judul_gabungan_en + lead_paragraph_gabungan_en
tanggal: [datetime.date(2021, 9, 1), datetime.date(2021, 9, 2)]
panjang token: [315, 225] (maks 512, lebih dari itu dipotong)
bentuk CLS: (2, 768) | mean: (2, 768)
ada NaN: False
cos(hari1, hari2)  CLS: 0.554 | mean: 0.88


## Embed semua hari, simpan, lalu evaluasi

Satu vektor per hari dari judul + lead paragraf (konten penuh terlalu panjang untuk batas 512 token). Hari tanpa berita diberi vektor nol dan penanda `has_news`. Dua pooling dicoba (`cls` dan `mean`). Embedding 768 dimensi diperkecil dengan PCA (20 komponen, di-fit hanya di train) supaya tidak overfit pada ~825 hari train.

In [21]:
# Embed SEMUA hari (jalankan sekali; hasil disimpan di data/features_c/ dan dipakai ulang).
# Di CPU ini bisa memakan waktu beberapa menit. Ubah LIMIT ke angka kecil (mis. 20) untuk uji cepat.
LIMIT = None
FEATURES_C.mkdir(parents=True, exist_ok=True)
f_cls, f_mean = FEATURES_C / "c2_finbert_cls.npy", FEATURES_C / "c2_finbert_mean.npy"

TITLE = "judul_gabungan" if "judul_gabungan" in text.columns else "judul_gabungan_en"
LEAD = "lead_paragraph_gabungan" if "lead_paragraph_gabungan" in text.columns else "lead_paragraph_gabungan_en"
day_text = (text[TITLE].fillna("") + ". " + text[LEAD].fillna("")).str.strip(" .")
has_news = (day_text.str.len() > 0).values

if LIMIT is None and f_cls.exists() and f_mean.exists():
    emb = {"cls": np.load(f_cls), "mean": np.load(f_mean)}
    print("dimuat dari cache:", emb["cls"].shape)
else:
    idx = np.where(has_news)[0][:LIMIT] if LIMIT else np.where(has_news)[0]
    out = {"cls": np.zeros((len(text), 768), dtype="float32"),   # hari tanpa berita = vektor nol
           "mean": np.zeros((len(text), 768), dtype="float32")}
    for i in range(0, len(idx), 16):
        b = idx[i:i+16]
        enc = tok(day_text.iloc[b].tolist(), padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        out["cls"][b] = h[:, 0].numpy()
        out["mean"][b] = ((h * m).sum(1) / m.sum(1)).numpy()
    emb = out
    if LIMIT is None:                       # jangan simpan cache kalau hanya uji sebagian
        np.save(f_cls, emb["cls"]); np.save(f_mean, emb["mean"])
        print("disimpan ke", FEATURES_C)
print("hari:", len(text), "| dengan berita:", int(has_news.sum()), "| NaN:", bool(np.isnan(emb["cls"]).any()))

disimpan ke ..\data\features_c
hari: 1202 | dengan berita: 1182 | NaN: False


In [23]:
# Gabungkan fitur ke D, lalu nilai di VALIDATION saja (test tidak disentuh).
# PCA di-fit HANYA di data train (hindari kebocoran), lalu diterapkan ke val dan test.
from sklearn.decomposition import PCA

N_COMP = 20
C2_COLS = {}
for pool in ["cls", "mean"]:
    feats = pd.DataFrame(emb[pool], index=text.index)
    train_idx = D["train"]["date"]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[train_idx].values)
    comp = pd.DataFrame(pca.transform(feats.values), index=text.index,
                        columns=[f"c2_{pool}_{i}" for i in range(N_COMP)])
    comp[f"c2_{pool}_has_news"] = has_news.astype(float)
    for s in D:
        D[s] = D[s].drop(columns=[c for c in comp.columns if c in D[s].columns]) \
                   .merge(comp, left_on="date", right_index=True, how="left")
    C2_COLS[pool] = list(comp.columns)
    print(pool, "variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))

# n_configs_tried=2 karena kami mencoba 2 pooling (cls, mean) -> dilaporkan jujur di laporan.
for pool in ["cls", "mean"]:
    evaluate_val(f"C2_finbert_{pool}_LR", "LR", D, PRICE + C2_COLS[pool], n_configs_tried=2,
                 note=f"FinBERT {pool} pooling, judul+lead per hari, PCA {N_COMP}")
# Setelah memilih varian terbaik di validation, jalankan final_test(...) SEKALI untuk varian itu saja.

cls variansi dijelaskan PCA: 0.789
mean variansi dijelaskan PCA: 0.717
C2_finbert_cls_LR [LR] val: DA=0.494, BalAcc=0.494, F1_macro=0.494, MCC=-0.012, AUC=0.461
C2_finbert_mean_LR [LR] val: DA=0.483, BalAcc=0.482, F1_macro=0.482, MCC=-0.036, AUC=0.486
